In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import json 
import os

In [ ]:
obj_id = 'AT2026brq'
z = 0.04586 ## estimate from simbad suspected host galaxy https://simbad.u-strasbg.fr/simbad/sim-coo?Coord=151.17226968482592%202.772519570258005&Radius.unit=arcsec&Radius=10
obj = pd.read_csv(f'data/{obj_id}/forced_photometry.csv')
obj.head()

,oid,survey_id,measurement_id,mjd,ra,dec,band,band_name,psfFlux,psfFluxErr,...,procstatus,programid,ranr,rcid,rfid,scibckgnd,sciinpseeing,scisigpix,sharpnr,sigmagnr
0,313888627043074080,lsst,170019716343005213,61088.211979,151.172266,2.772515,2,r,4468.2030,166.28058,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,313888627043074080,lsst,170019716473028612,61088.212444,151.172266,2.772515,2,r,4541.0103,162.41673,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,313888627043074080,lsst,170019717148311684,61088.217394,151.172266,2.772515,4,z,6152.7036,388.87543,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,313888627043074080,lsst,170019717283053580,61088.217859,151.172266,2.772515,4,z,6249.4030,366.21268,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,313888627043074080,lsst,170028510526046300,61090.180598,151.172267,2.772515,3,i,5809.4395,277.43387,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def flux2mag(flux, flux_err):
    mag = -2.5 * np.log10(flux) + 31.4 ## zero point according to https://dp1.lsst.io/tutorials/portal/103/portal-103-3.html
    mag_err = 2.5 / np.log(10) * flux_err / flux
    return mag, mag_err

mag, mag_err = flux2mag(obj['psfFlux'], obj['psfFluxErr'])

/opt/anaconda3/envs/babamul/lib/python3.11/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [6]:
conv_obj = pd.DataFrame()
conv_obj['id'] = obj['oid']
conv_obj['mjd'] = obj['mjd']
conv_obj['mag'] = mag
conv_obj['mag_err'] = mag_err
conv_obj['limiting_mag'] = obj['diffmaglim']
conv_obj['filter'] = obj['band_name']
conv_obj['instrument_name'] = 'LSST'
conv_obj['snr'] = obj['psfFlux'] / obj['psfFluxErr']
conv_obj['magsys'] = 'ab'
conv_obj['flux'] = obj['psfFlux']
conv_obj['flux_err'] = obj['psfFluxErr']
conv_obj['instrument_id'] = obj['visit']
conv_obj.head()

,id,mjd,mag,mag_err,limiting_mag,filter,instrument_name,snr,magsys,flux,flux_err,instrument_id
0,313888627043074080,61088.211979,22.274668,0.040405,NaN,r,LSST,26.871466,ab,4468.2030,166.28058,2026021600255
1,313888627043074080,61088.212444,22.257119,0.038833,NaN,r,LSST,27.959006,ab,4541.0103,162.41673,2026021600256
2,313888627043074080,61088.217394,21.927335,0.068623,NaN,z,LSST,15.821785,ab,6152.7036,388.87543,2026021600261
3,313888627043074080,61088.217859,21.910404,0.063624,NaN,z,LSST,17.064955,ab,6249.4030,366.21268,2026021600262
4,313888627043074080,61090.180598,21.989664,0.051850,NaN,i,LSST,20.939907,ab,5809.4395,277.43387,2026021800241


In [7]:
conv_obj.to_csv(f'data/{obj_id}/{obj_id}_photometry.csv', index=False)